# 44 - SOCOFing / NBIS feasibility pilot

Run in Colab. Download the noncommercial research dataset from [Kaggle](https://www.kaggle.com/datasets/ruizgara/socofing) and put the original, unextracted ZIP at `MyDrive/FingerprintResearch/archives/socofing.zip`, or change `ARCHIVE` below. The ZIP can contain a mirrored `socofing/SOCOFing` tree; the pilot uses each image once and verifies the selected mirrored files byte for byte. If NBIS is missing, use Notebook 00 with `PREPARE_NBIS=True` (do not run Notebook 41). CPU is sufficient; a GPU is optional.

No training, threshold selection, impostor scoring or FVC evaluation. Fixed pilot of 12 people selected by hash before scoring; do not include these people in a future locked SOCOFing test. Compare raw, frozen Notebook 39 L1 and frozen L1+triplet. The unmodified original is the gallery; altered probes are derived from that same image, not independent impressions. Report gallery usability and probe/genuine-pair matchability by alteration and level, including failures. Do not interpret pilot matchability as FRR/FAR or as evidence for real dry fingerprints. Only aggregate outputs and provenance are saved to Drive; templates and logs stay in the Colab runtime.

In [ ]:
import hashlib, importlib.util, io, json, re, subprocess, tempfile, zipfile
from pathlib import Path, PurePosixPath

import numpy as np
import pandas as pd
import torch
from PIL import Image

VERSION = 'socofing_nbis_pilot_v1'
TRAIN_SHA = '59ad0c66eee00fe6e050cb02749083cc39ebb7af7a31a599a450c94d4e4a7d16'
VALIDATION_SHA = 'b1ddff0a86f722ea06634f784050b6d8cfb6a0e37a0a91f010acc499df6e4127'
LEVELS = ('easy', 'medium', 'hard')
KINDS = ('obl', 'cr', 'zcut')
METHODS = ('raw', 'native_l1', 'native_triplet')
PILOT_SUBJECTS = sorted(sorted(range(1, 601), key=lambda s: hashlib.sha256(f'{VERSION}:{s}'.encode()).digest())[:12])
NAME = re.compile(r'(\d+)__([MF])_(Left|Right)_(little|ring|middle|index|thumb)_finger(?:_(Obl|CR|Z-?cut))?\.bmp', re.I)

def file_digest(path):
    h = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def json_bytes(value):
    return (json.dumps(value, sort_keys=True, indent=2) + '\n').encode()

def write_verified(path, data):
    if path.exists():
        if path.read_bytes() != data:
            raise RuntimeError(f'Refusing to overwrite differing output: {path}')
        return
    pending = path.with_suffix(path.suffix + '.tmp')
    pending.write_bytes(data)
    pending.replace(path)
    if path.read_bytes() != data:
        raise RuntimeError(f'Write verification failed: {path}')

def checked_json(path, expected):
    if file_digest(path) != expected:
        raise RuntimeError(f'Pinned configuration differs: {path}')
    return json.loads(path.read_bytes())

def index_archive(archive, subjects):
    originals, altered = {}, {}
    seen_paths = set()
    tags = {f'altered-{level}': level for level in LEVELS}
    for item in archive.infolist():
        path = PurePosixPath(item.filename)
        if item.is_dir() or path.suffix.lower() != '.bmp' or path.parts[0] == '__MACOSX':
            continue
        match = NAME.fullmatch(path.name)
        parent = path.parent.name.lower()
        if not match or parent not in ('real', *tags):
            raise RuntimeError(f'Unexpected SOCOFing BMP: {item.filename}')
        subject, sex, hand, finger, kind = match.groups()
        key = (int(subject), hand.lower(), finger.lower())
        if key[0] not in range(1, 601) or (kind is None) != (parent == 'real'):
            raise RuntimeError(f'Unexpected SOCOFing label: {item.filename}')
        target, label = (originals, key) if kind is None else (altered, (key, tags[parent], kind.lower().replace('-', '')))
        if item.filename in seen_paths:
            raise RuntimeError(f'Duplicate ZIP member path: {item.filename}')
        seen_paths.add(item.filename)
        if label in target:
            prior, _ = target[label]
            canonical, mirror = sorted((prior, item), key=lambda member: len(member.filename))
            if (not canonical.filename.startswith('SOCOFing/')
                    or mirror.filename != 'socofing/' + canonical.filename
                    or (canonical.file_size, canonical.CRC) != (mirror.file_size, mirror.CRC)
                    or (key[0] in subjects and archive.read(canonical) != archive.read(mirror))):
                raise RuntimeError(f'Conflicting duplicate SOCOFing images: {prior.filename} and {item.filename}')
            item = canonical
        target[label] = (item, sex.lower())
    if len(originals) != 6000 or any(key not in originals or originals[key][1] != sex for (key, _, _), (_, sex) in altered.items()):
        raise RuntimeError('Expected the complete original SOCOFing gallery and a matching original for every altered BMP.')
    gallery = {key: item for key, (item, _) in originals.items() if key[0] in subjects}
    probes = {label: item for label, (item, _) in altered.items() if label[0][0] in subjects}
    if len(gallery) != 10 * len(subjects) or {(level, kind) for _, level, kind in probes} != {(l, k) for l in LEVELS for k in KINDS}:
        raise RuntimeError('Incomplete selected gallery or missing pilot alteration stratum.')
    return gallery, probes

def image_pixels(archive, member):
    with Image.open(io.BytesIO(archive.read(member))) as image:
        if image.format != 'BMP' or image.size != (96, 103):
            raise RuntimeError(f'Unexpected SOCOFing image format or size: {member.filename}')
        return np.array(image.convert('L'), dtype=np.uint8)

def summarize(scored):
    result = scored.groupby(['level', 'alteration', 'method'], sort=True).agg(
        probes=('matchable', 'size'), probe_usable=('probe_usable', 'sum'),
        gallery_usable=('gallery_usable', 'sum'), matchable=('matchable', 'sum'),
        median_probe_minutiae=('minutiae', 'median')).reset_index()
    if len(result) != len(LEVELS) * len(KINDS) * len(METHODS):
        raise RuntimeError('Incomplete pilot results.')
    result['unmatchable_pct'] = 100 * (result['probes'] - result['matchable']) / result['probes']
    return result

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/FingerprintResearch')
ARCHIVE = PROJECT / 'archives' / 'socofing.zip'
NBIS = Path('/content/fingerprint_research/tools/nbis_5_0_0/install/bin')
MINDTCT, BOZORTH3 = (NBIS / name for name in ('mindtct', 'bozorth3'))
if not ARCHIVE.is_file():
    raise FileNotFoundError(f'Put the unextracted Kaggle SOCOFing ZIP at {ARCHIVE}.')
if not all(path.is_file() for path in (MINDTCT, BOZORTH3)):
    raise FileNotFoundError('NBIS missing. Run Notebook 00 with PREPARE_NBIS=True, then return here.')
validation_dir = PROJECT / 'evaluations' / 'native_triplet_replication_validation_v1' / VALIDATION_SHA[:20]
train_dir = PROJECT / 'experiments' / 'native_triplet_replication_v1' / TRAIN_SHA[:20]
config = checked_json(validation_dir / 'configuration.json', VALIDATION_SHA)
train = checked_json(train_dir / 'configuration.json', TRAIN_SHA)
binary_sha = {path.name: file_digest(path) for path in (MINDTCT, BOZORTH3)}
if binary_sha != config['nbis_binary_sha256'] or config['pilot_configuration_sha256'] != TRAIN_SHA:
    raise RuntimeError('NBIS binaries or training provenance differ from Notebook 40.')
versions = {'torch_version': str(torch.__version__), 'torch_cuda': str(torch.version.cuda),
            'numpy_version': np.__version__, 'pillow_version': Image.__version__}
if any(config[key] != value or train[key] != value for key, value in versions.items()):
    raise RuntimeError('Runtime versions differ from Notebook 40. Restore them with Notebook 00.')
m0 = PROJECT / 'experiments' / 'm0_unet_v1_seed20261110'
model_path = m0 / 'restoration_baseline.py'
model_sha = file_digest(model_path)
if model_sha != json.loads((m0 / 'configuration.json').read_bytes())['source_file_sha256'][model_path.name]:
    raise RuntimeError('Restoration model source checksum differs.')
spec = importlib.util.spec_from_file_location('socofing_restorer', model_path)
if spec is None or spec.loader is None:
    raise RuntimeError('Cannot load the verified restoration model source.')
training = importlib.util.module_from_spec(spec)
spec.loader.exec_module(training)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
models = {}
for method, epoch in zip(METHODS[1:], (1, 2)):
    pointer = config['model_checkpoints'][method]
    path = train_dir / method / 'selected.pt'
    if (pointer['file'] != path.name or pointer['completed_epochs'] != epoch
            or pointer['configuration_sha256'] != TRAIN_SHA or file_digest(path) != pointer['sha256']):
        raise RuntimeError(f'Checkpoint differs: {method}')
    state = torch.load(path, map_location='cpu', weights_only=True)
    if (state['run'], state['completed_epochs'], state['configuration_sha256']) != (method, epoch, TRAIN_SHA):
        raise RuntimeError(f'Checkpoint payload differs: {method}')
    model = training.RestorationUNet()
    model.load_state_dict(state['model'], strict=True)
    models[method] = model.to(DEVICE).eval()
helper = validation_dir / 'frozen_helpers.py.txt'
if file_digest(helper) != config['frozen_helpers_sha256']:
    raise RuntimeError('Notebook 40 NBIS helper checksum differs.')
print('Verified fixed checkpoints and NBIS:', DEVICE, '| pilot subjects:', len(PILOT_SUBJECTS))

In [ ]:
import ast

def evaluate(archive, gallery, probes, models, helpers, work):
    galleries = {str(key): helpers['extract_png'](image_pixels(archive, item)) for key, item in sorted(gallery.items())}
    pairs = pd.DataFrame([{'probe_path': item.filename, 'gallery_path': str(key), 'level': level, 'alteration': kind}
                          for (key, level, kind), item in sorted(probes.items())])
    helpers['pairs'] = pairs
    rows = {method: [] for method in METHODS}
    with torch.inference_mode():
        for number, item in enumerate((probes[key] for key in sorted(probes)), 1):
            pixels = image_pixels(archive, item)
            image = torch.from_numpy(pixels).to(device=DEVICE, dtype=torch.float32)[None, None] / 255
            for method in METHODS:
                if method == 'raw':
                    result = pixels
                else:
                    prediction = models[method](image)
                    if prediction.shape != image.shape or not torch.isfinite(prediction).all():
                        raise RuntimeError(f'Invalid restoration output: {method}, {item.filename}')
                    result = helpers['quantize'](prediction[0, 0].cpu().numpy())
                rows[method].append({'relative_path': item.filename, **helpers['extract_png'](result)})
            if number % 100 == 0:
                print(f'Extracted {number}/{len(probes)} altered probes for all three methods.', flush=True)
    gallery_frame = pd.DataFrame([{'relative_path': key, **value} for key, value in galleries.items()])
    scored = []
    for method, records in rows.items():
        match_dir = work / method
        match_dir.mkdir()
        probe_frame = pd.DataFrame(records)
        pairs_scored = helpers['match_pairs'](probe_frame, gallery_frame, match_dir)
        scored.append(pairs_scored.assign(method=method, probe_usable=pairs_scored.probe_status.eq('ok'),
            gallery_usable=pairs_scored.gallery_status.eq('ok'),
            minutiae=pairs_scored.probe_path.map(probe_frame.set_index('relative_path').minutiae)))
    return summarize(pd.concat(scored, ignore_index=True)), int(gallery_frame.status.eq('ok').sum())

archive_sha = file_digest(ARCHIVE)
with zipfile.ZipFile(ARCHIVE) as archive, tempfile.TemporaryDirectory(prefix='socofing44-') as scratch:
    gallery, probes = index_archive(archive, PILOT_SUBJECTS)
    work = Path(scratch)
    template_dir = work / 'templates'
    template_dir.mkdir()
    names = {'quantize', 'read_xyt', 'extract_png', 'match_pairs'}
    module = ast.parse(helper.read_text(encoding='utf-8'), filename=str(helper))
    module.body = [node for node in module.body if isinstance(node, ast.FunctionDef) and node.name in names]
    if len(module.body) != len(names) or {node.name for node in module.body} != names:
        raise RuntimeError('Verified NBIS helper does not expose the expected functions.')
    helpers = dict(globals(), TEMPLATE_DIR=template_dir)
    exec(compile(module, str(helper), 'exec'), helpers)
    summary, gallery_usable = evaluate(archive, gallery, probes, models, helpers, work)

provenance = dict(version=VERSION, archive_sha256=archive_sha, validation_sha256=VALIDATION_SHA,
    training_sha256=TRAIN_SHA, checkpoints=config['model_checkpoints'], model_source_sha256=model_sha,
    nbis_binary_sha256=binary_sha, versions=versions, device=str(DEVICE), pilot_subjects=PILOT_SUBJECTS,
    native_size=[96, 103], png_dpi=500, level_order=list(LEVELS),
    galleries=len(gallery), galleries_usable=gallery_usable, altered_probes=len(probes))
data = {'configuration.json': json_bytes(provenance), 'pilot_matchability.csv': summary.to_csv(index=False).encode()}
run_id = hashlib.sha256(data['configuration.json']).hexdigest()[:20]
report = PROJECT / 'evaluations' / VERSION / run_id
report.mkdir(parents=True, exist_ok=True)
for name, content in data.items():
    write_verified(report / name, content)
write_verified(report / 'completion.json', json_bytes({'status': 'socofing_pilot_complete',
    'file_sha256': {name: hashlib.sha256(content).hexdigest() for name, content in data.items()}}))
print(f'Usable original galleries: {gallery_usable}/{len(gallery)}')
print(summary.to_string(index=False, float_format=lambda x: f'{x:.2f}'))
print('SOCOFING PILOT COMPLETE:', report)